# New run: push a config.json

Builds a `RunConfig` (validated by `run_config.RunConfig`) for one run and pushes it to `runs/{run_id}/config.json` on the `trainvols` volume -- the notebook equivalent of `push_config.py`.

Example below: two jobs, `ETL` and `Count`, where `Count` depends on `ETL` finishing first. job_uid IS the class name in `jobs.py` -- `RunConfig` checks that at construction, so a typo or a renamed class fails right here, not when `main.py` tries to launch it.

In [20]:
import io
import time

import modal

from config import VOLUME_NAME
from jobs import preflight_check
from run_config import JobEntry, ResourcesSpec, RunConfig
from main import volume

## Build and validate the config

Edit `run_id` and the job entries below. Construction itself is the validation -- a bad type, shape, or job_uid raises right here, not three cells later.

In [21]:
run_id = "count-etl-demo"  # change per run

config = RunConfig(
    metadata={"run_id": run_id, "pushed_ts": time.time()},
    jobs={
        "ETL": JobEntry(
            resources=ResourcesSpec(cpu=2),
        ),
        "Count": JobEntry(
            dependencies=["ETL"],
        ),
    },
)
config

RunConfig(metadata={'run_id': 'count-etl-demo', 'pushed_ts': 1787074964.816399}, jobs={'ETL': JobEntry(parameters={}, dependencies=[], resources=ResourcesSpec(cpu=2.0, gpu_type=None, gpu_count=None)), 'Count': JobEntry(parameters={}, dependencies=['ETL'], resources=ResourcesSpec(cpu=None, gpu_type=None, gpu_count=None))})

In [24]:
raw = b"".join(volume.read_file(f"runs/{run_id}/config.json"))
raw

b'{\n  "metadata": {\n    "run_id": "count-etl-demo",\n    "pushed_ts": 1787074964.816399\n  },\n  "jobs": {\n    "ETL": {\n      "parameters": {},\n      "dependencies": [],\n      "resources": {\n        "cpu": 2.0,\n        "gpu_type": null,\n        "gpu_count": null\n      }\n    },\n    "Count": {\n      "parameters": {},\n      "dependencies": [\n        "ETL"\n      ],\n      "resources": {\n        "cpu": null,\n        "gpu_type": null,\n        "gpu_count": null\n      }\n    }\n  }\n}'

In [25]:
RunConfig.model_validate_json(raw)

RunConfig(metadata={'run_id': 'count-etl-demo', 'pushed_ts': 1787074964.816399}, jobs={'ETL': JobEntry(parameters={}, dependencies=[], resources=ResourcesSpec(cpu=2.0, gpu_type=None, gpu_count=None)), 'Count': JobEntry(parameters={}, dependencies=['ETL'], resources=ResourcesSpec(cpu=None, gpu_type=None, gpu_count=None))})

In [16]:
preflight_check("count-etl-demo", volume)

{}

In [4]:
run_id = "transformer"  # change per run

config = RunConfig(
    metadata={"run_id": run_id, "pushed_ts": time.time()},
    jobs={
        "ETL": JobEntry(
            resources=ResourcesSpec(cpu=2),
        ),
        "Count": JobEntry(
            dependencies=["ETL"],
        ),
    },
)
config

RunConfig(metadata={'run_id': 'transformer', 'pushed_ts': 1787072124.1297169}, jobs={'ETL': JobEntry(parameters={}, dependencies=[], resources=ResourcesSpec(cpu=2.0, gpu_type=None, gpu_count=None)), 'Count': JobEntry(parameters={}, dependencies=['ETL'], resources=ResourcesSpec(cpu=None, gpu_type=None, gpu_count=None))})

## Push it to the volume

Refuses to overwrite an existing `runs/{run_id}/config.json` unless `force = True` -- same rule `push_config.py` follows.

In [23]:
force = True

volume = modal.Volume.from_name(VOLUME_NAME, create_if_missing=True)
remote_path = f"runs/{run_id}/config.json"
payload = config.model_dump_json(indent=2).encode()

try:
    with volume.batch_upload(force=force) as batch:
        batch.put_file(io.BytesIO(payload), remote_path)
    print(f"pushed config to {remote_path}")
except FileExistsError:
    print(f"{remote_path} already exists -- set force = True to overwrite")

pushed config to runs/count-etl-demo/config.json


## Launch it

```
uv run modal run main.py::launch_job --run-id count-etl-demo --job ETL
uv run modal run main.py::launch_job --run-id count-etl-demo --job Count
```

`Count` refuses to launch until `ETL` has left its artifact behind. No registration step needed elsewhere -- `main.py` resolves `job_uid` straight to the `jobs.py` class of that name.